# End-to-End Claims Orchestration — Happy Path

In the previous notebook, the Intake Orchestrator handled **Phase 1 (Pre-Processing)** only. Now we extend it to run the **full claim lifecycle** in a single invocation:

| Phase | What Happens | Who Does It |
|---|---|---|
| **Phase 1: Pre-Processing** | Retrieve docs, authenticate, extract, verify policy | Pre-processing graph (Authenticator, Extractor, Policy Verification) |
| **Phase 2: Adjudication** | Apply business rules to the verification decision | Supervisor via `adjudicate_claim` tool (rule-based, no human) |
| **Phase 3: Communication** | Draft and send the claim decision notification | Communicator Agent via `send_claim_decision` tool |

This is the **happy path** — a clean claim with no flags, no exclusions, and consistent cross-document data. The supervisor auto-approves and the Communicator drafts the notification.

### What About Claims That Need Human Review?

When the adjudication tool detects flags (exclusions triggered, identity confidence below threshold, cross-document inconsistencies), it would route the claim to a human-in-the-loop workflow instead of auto-approving. That pattern is covered in the optional **Human-in-the-Loop with Step Functions** notebook.

### Pipeline Overview

```
Supervisor Agent (Intake Orchestrator)
  │
  ├── retrieve_claim_documents       ← S3
  ├── run_preprocessing_graph        ← Authenticator → Extractor → Policy Verification
  ├── adjudicate_claim               ← Rule-based auto-decision
  ├── send_claim_decision            ← Communicator Agent
  └── persist_claim_to_dynamodb      ← DynamoDB (final record)
```

## Setup

Same setup as previous notebooks. We also import the new **Communicator Agent** module from the agents/ directory.

In [ ]:
import boto3
import json
import sys
import os
import re
from pathlib import Path
from datetime import datetime
from decimal import Decimal

from strands import Agent, tool
from strands.models import BedrockModel
from strands.multiagent import GraphBuilder
from strands.tools.mcp import MCPClient
from mcp import stdio_client, StdioServerParameters

# ── Path setup ──────────────────────────────────────────────────────────────────
_cwd = Path.cwd()
REPO_ROOT = _cwd
for _parent in [_cwd] + list(_cwd.parents):
    if (_parent / "agents").is_dir() and (_parent / "mcp_servers").is_dir():
        REPO_ROOT = _parent
        break
sys.path.insert(0, str(REPO_ROOT))

MCP_SERVER_PATH = REPO_ROOT / "mcp_servers" / "socotra_mock"
MOCK_DATA_PATH = MCP_SERVER_PATH / "mock_data.json"

# ── Import shared agent modules ──────────────────────────────────────────────────
from agents import authenticator_agent
from agents import extractor_agent
from agents import policy_verification_agent
from agents import communicator_agent

# ── AWS configuration ────────────────────────────────────────────────────────────
REGION = "us-east-1"
S3_BUCKET = os.environ["WORKSHOP_S3_BUCKET"]  # Set by lifecycle config or export manually
S3_PREFIX = "claims-processing/claimant-data/"
DYNAMODB_TABLE = "claims-metadata"

s3_client = boto3.client("s3", region_name=REGION)
dynamodb = boto3.resource("dynamodb", region_name=REGION)

print("✅ Setup complete")
print(f"   REPO_ROOT  : {REPO_ROOT}")
print(f"   S3 bucket  : s3://{S3_BUCKET}/{S3_PREFIX}")
print(f"   DynamoDB   : {DYNAMODB_TABLE}")

## The Communicator Agent

The Communicator Agent is a shared module in `agents/communicator_agent.py`. It drafts claim decision notifications to beneficiaries. Let’s look at how it’s configured:

In [ ]:
# Peek at the Communicator Agent module
import inspect
print(inspect.getsource(communicator_agent))

## Define the Supervisor’s Tools

The supervisor now has **five** tools — the three from notebook 01 plus two new ones for adjudication and communication:

| Tool | Phase | What It Does |
|---|---|---|
| `retrieve_claim_documents` | Pre-Processing | Downloads claim PDFs from S3 |
| `run_preprocessing_graph` | Pre-Processing | Runs authenticate → extract → verify graph |
| `adjudicate_claim` | Adjudication | Applies business rules to verification output |
| `send_claim_decision` | Communication | Invokes Communicator Agent to draft notification |
| `persist_claim_to_dynamodb` | All | Writes the final claim record to DynamoDB |

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Tool 1: Retrieve claim documents from S3 (same as notebook 01_multi_agent_orchestration)
# ─────────────────────────────────────────────────────────────────────────────

LOCAL_DOCS_DIR = Path("claim_documents")
LOCAL_DOCS_DIR.mkdir(exist_ok=True)

@tool
def retrieve_claim_documents(s3_bucket: str, s3_prefix: str) -> str:
    """Download claim PDF documents from S3.

    Args:
        s3_bucket: S3 bucket name
        s3_prefix: S3 key prefix for the claim documents
    """
    response = s3_client.list_objects_v2(Bucket=s3_bucket, Prefix=s3_prefix)
    s3_objects = response.get("Contents", [])
    downloaded = []
    for obj in s3_objects:
        key = obj["Key"]
        filename = key.split("/")[-1]
        if not filename or not filename.endswith(".pdf"):
            continue
        local_path = LOCAL_DOCS_DIR / filename
        s3_client.download_file(s3_bucket, key, str(local_path))
        downloaded.append(str(local_path))
        print(f"   ✅ {filename}")
    return json.dumps({"status": "success", "documents_retrieved": len(downloaded), "document_paths": downloaded})


print("✅ retrieve_claim_documents tool defined")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Tool 2: Pre-processing graph (same as notebook 01_multi_agent_orchestration)
# ─────────────────────────────────────────────────────────────────────────────

SOCOTRA_SERVER_SCRIPT = str(MCP_SERVER_PATH / "server.py")

socotra_mcp_client = MCPClient(
    lambda: stdio_client(
        StdioServerParameters(
            command=sys.executable,
            args=[SOCOTRA_SERVER_SCRIPT],
            env={"MOCK_DATA_PATH": str(MOCK_DATA_PATH), **os.environ}
        )
    )
)

auth_passed = True

@tool
def run_preprocessing_graph(claim_prompt: str) -> str:
    """Run the 3-node pre-processing graph: authenticate, extract, verify_policy.

    Args:
        claim_prompt: Full claim submission prompt with all claim details
    """
    global auth_passed
    auth_passed = True
    mcp_tools = socotra_mcp_client.list_tools_sync()
    auth_agent = authenticator_agent.build_agent(mcp_tools)
    output_dir = Path("extracted_output")
    output_dir.mkdir(exist_ok=True)
    extract_agent = extractor_agent.build_agent(output_dir=output_dir)
    verify_agent = policy_verification_agent.build_agent(mcp_tools)

    def check_auth_passed(state):
        return auth_passed

    builder = GraphBuilder()
    builder.add_node(auth_agent, "authenticate")
    builder.add_node(extract_agent, "extract")
    builder.add_node(verify_agent, "verify_policy")
    builder.set_entry_point("authenticate")
    builder.add_edge("authenticate", "extract", condition=check_auth_passed)
    builder.add_edge("extract", "verify_policy")
    graph = builder.build()
    result = graph(claim_prompt)
    return str(result)


print("✅ run_preprocessing_graph tool defined")

### New Tool: `adjudicate_claim`

This is a **rule-based** tool, not another agent. The supervisor already has the full verification decision from the pre-processing graph. The adjudication tool applies business rules to decide:

- **AUTO-APPROVE**: Verification passed, no exclusions, no flags, documents consistent
- **ESCALATE**: Any flags, exclusions, or inconsistencies detected → would route to human review (covered in human in the loop notebook)

In production, the escalation path would trigger a Step Functions workflow or Quick Automate case for human adjudication. In this notebook, we only handle the happy path (auto-approve).

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Tool 3: Adjudicate claim (rule-based auto-decision)
# ─────────────────────────────────────────────────────────────────────────────

@tool
def adjudicate_claim(
    claim_id: str,
    verification_summary: str,
    auth_summary: str,
) -> str:
    """Apply business rules to the verification output and return an adjudication decision.

    Args:
        claim_id: The claim identifier
        verification_summary: Full verification decision from the pre-processing graph
        auth_summary: Authentication result summary
    """
    # --- Parse verification decision ---
    # Strategy: try JSON extraction first, fall back to keyword scanning
    decision = {}
    # Try to find a JSON block with verification_decision
    json_blocks = re.findall(r'\{[^{}]*(?:\{[^{}]*\}[^{}]*)*\}', verification_summary)
    for block in json_blocks:
        if 'verification_decision' in block or 'overall_status' in block:
            try:
                parsed = json.loads(block)
                decision = parsed
                break
            except json.JSONDecodeError:
                continue

    vd = decision.get('verification_decision', decision)
    overall = vd.get("overall_status", "")
    exclusions = vd.get("exclusion_check", {}).get("exclusions_triggered", [])
    flags = vd.get("flags", [])
    consistency = vd.get("cross_document_consistency", {}).get("status", "")

    # Fallback: scan text for keyword signals if JSON parsing yielded nothing
    combined_text = (verification_summary + ' ' + auth_summary).upper()
    if not overall:
        if "VERIFIED" in combined_text and "DENIED" not in combined_text:
            overall = "VERIFIED"
        elif "DENIED" in combined_text or "FAILED" in combined_text:
            overall = "DENIED"
    if not consistency:
        if "INCONSISTENC" in combined_text:
            consistency = "INCONSISTENCIES_FOUND"
        elif "CONSISTENT" in combined_text:
            consistency = "CONSISTENT"
    if not exclusions:
        exclusions = []  # No exclusions found — do not scan text (too many false positives)

    # Business rules
    is_verified = overall == "VERIFIED"
    is_consistent = consistency in ("CONSISTENT", "")
    no_exclusions = not exclusions
    no_flags = not flags

    if is_verified and no_exclusions and no_flags and is_consistent:
        adjudication_decision = "APPROVED"
        rationale = "All checks passed: verification VERIFIED, no exclusions, no flags, documents consistent."
    else:
        adjudication_decision = "ESCALATED"
        reasons = []
        if not is_verified:
            reasons.append(f"Verification status: {overall or 'UNKNOWN'}")
        if exclusions:
            reasons.append(f"Exclusions triggered: {exclusions}")
        if flags:
            reasons.append(f"Flags: {flags}")
        if not is_consistent:
            reasons.append(f"Cross-document consistency: {consistency}")
        rationale = "Escalation required: " + "; ".join(reasons)

    print(f"   🏛️ Adjudication for {claim_id}: {adjudication_decision}")
    print(f"   Rationale: {rationale}")

    return json.dumps({
        "claim_id": claim_id,
        "adjudication_decision": adjudication_decision,
        "rationale": rationale,
        "adjudicated_at": datetime.now(datetime.UTC).isoformat() + "Z",
    })


print("✅ adjudicate_claim tool defined")

### New Tool: `send_claim_decision`

This tool wraps the **Communicator Agent** from `agents/communicator_agent.py`. The supervisor passes the claim decision and beneficiary contact info, and the Communicator drafts a formal notification letter.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Tool 4: Send claim decision via Communicator Agent
# ─────────────────────────────────────────────────────────────────────────────

@tool
def send_claim_decision(
    claim_id: str,
    claimant_name: str,
    policy_number: str,
    adjudication_decision: str,
    benefit_amount: str,
    adjudication_notes: str,
    beneficiary_email: str,
) -> str:
    """Draft and send a claim decision notification using the Communicator Agent.

    Args:
        claim_id: The claim identifier
        claimant_name: Beneficiary name
        policy_number: Policy number
        adjudication_decision: APPROVED, DENIED, or ESCALATED
        benefit_amount: Payable benefit amount
        adjudication_notes: Rationale for the decision
        beneficiary_email: Beneficiary email address
    """
    comm_agent = communicator_agent.build_agent()

    prompt = (
        f"Draft a claim decision notification:\n\n"
        f"Claim ID: {claim_id}\n"
        f"Claimant: {claimant_name}\n"
        f"Policy: {policy_number}\n"
        f"Decision: {adjudication_decision}\n"
        f"Benefit Amount: {benefit_amount}\n"
        f"Notes: {adjudication_notes}\n"
        f"Beneficiary Email: {beneficiary_email}\n"
    )

    result = comm_agent(prompt)
    notification_text = str(result)

    print(f"   ✉️  Notification drafted for {claimant_name} ({beneficiary_email})")
    print(f"   Decision: {adjudication_decision}")

    return json.dumps({
        "status": "notification_drafted",
        "claim_id": claim_id,
        "recipient": beneficiary_email,
        "decision": adjudication_decision,
        "notification_preview": notification_text[:500],
    })


print("✅ send_claim_decision tool defined")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Tool 5: Persist claim to DynamoDB (extended with adjudication fields)
# ─────────────────────────────────────────────────────────────────────────────

@tool
def persist_claim_to_dynamodb(
    claim_id: str,
    policy_number: str,
    claimant_name: str,
    date_of_death: str,
    auth_summary: str,
    verification_summary: str,
    adjudication_decision: str,
    adjudication_notes: str,
    notification_status: str,
) -> str:
    """Write the final claim record to DynamoDB with all phase results.

    Args:
        claim_id: Unique claim identifier
        policy_number: Policy number
        claimant_name: Name of the claimant
        date_of_death: Date of death (YYYY-MM-DD)
        auth_summary: Authentication result summary
        verification_summary: Policy verification result summary
        adjudication_decision: APPROVED, DENIED, or ESCALATED
        adjudication_notes: Adjudication rationale
        notification_status: Whether notification was sent
    """
    from botocore.exceptions import ClientError

    claim_record = {
        "claim_id": claim_id,
        "policy_number": policy_number,
        "claimant_name": claimant_name,
        "date_of_death": date_of_death,
        "auth_result_summary": auth_summary[:2000],
        "verification_summary": verification_summary[:3000],
        "adjudication_decision": adjudication_decision,
        "adjudication_notes": adjudication_notes,
        "notification_status": notification_status,
        "created_at": datetime.now(datetime.UTC).isoformat() + "Z",
        "stage": "pipeline_complete",
        "pipeline": "intake_orchestrator_e2e_v1",
    }

    try:
        table = dynamodb.create_table(
            TableName=DYNAMODB_TABLE,
            KeySchema=[{"AttributeName": "claim_id", "KeyType": "HASH"}],
            AttributeDefinitions=[{"AttributeName": "claim_id", "AttributeType": "S"}],
            BillingMode="PAY_PER_REQUEST",
        )
        table.wait_until_exists()
    except ClientError as e:
        if e.response["Error"]["Code"] == "ResourceInUseException":
            table = dynamodb.Table(DYNAMODB_TABLE)
        else:
            raise

    table.put_item(Item=claim_record)
    return json.dumps({"status": "success", "claim_id": claim_id, "stage": "pipeline_complete"})


print("✅ persist_claim_to_dynamodb tool defined")

## Define the Supervisor Agent

The Intake Orchestrator now has five tools spanning all three phases. The system prompt instructs it to run the full pipeline in sequence.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Define the Intake Orchestrator (End-to-End Supervisor)
# Note the edited workflow description and guidelines in the system prompt
# ─────────────────────────────────────────────────────────────────────────────

SUPERVISOR_SYSTEM_PROMPT = """
You are the Intake Orchestrator — the supervisor agent for an Insurance Claims Processing workflow.

## Your Role:
You coordinate the full claim lifecycle across three phases.

## WorkflowWhen you receive a claim submission:
1. Call retrieve_claim_documents to download the claim documents from S3
2. Build a detailed prompt with all claim details and the EXACT document_paths returned by retrieve_claim_documents. Do NOT construct filenames yourself.
3. Call run_preprocessing_graph with that prompt
4. Review the graph output for authentication status and verification decision
5. Call adjudicate_claim with the claim_id, verification_summary, and auth_summary
6. If the adjudication decision is APPROVED, call send_claim_decision with the claim details and beneficiary email
7. Call persist_claim_to_dynamodb with ALL results from every phase
8. Summarize what you completed across all three phases

## Guidelines
- Always run all phases in order — do not skip steps
- If adjudication returns ESCALATED, still persist the record but note that human review is pending
- In your final summary, report what YOU completed — do not echo specialist agent recommendations
- Include the adjudication decision and whether the notification was sent
"""

supervisor_model = BedrockModel(
    model_id="us.amazon.nova-2-lite-v1:0",
    region_name=REGION,
    additional_request_fields={
        "reasoningConfig": {"type": "enabled", "maxReasoningEffort": "low"}
    },
)

intake_orchestrator = Agent(
    model=supervisor_model,
    system_prompt=SUPERVISOR_SYSTEM_PROMPT,
    tools=[
        retrieve_claim_documents,
        run_preprocessing_graph,
        adjudicate_claim,
        send_claim_decision,
        persist_claim_to_dynamodb,
    ],
)

print("✅ Intake Orchestrator defined (end-to-end, 5 tools)")

## Run the Full Pipeline

This is the happy path — a clean claim that will auto-approve. The supervisor will:
1. Retrieve documents from S3
2. Run the pre-processing graph (authenticate → extract → verify)
3. Auto-adjudicate (all checks pass → APPROVED)
4. Draft a notification via the Communicator Agent
5. Persist the complete record to DynamoDB

In [ ]:
CLAIM_SUBMISSION = {
    "claim_id": "CLM-2026-00101",
    "policy_number": "WL-4582-1093",
    "claimant_name": "Lisa Doe",
    "date_of_death": "2026-01-15",
    "death_circumstances": "Natural causes — congestive heart failure at residence",
    "beneficiary_email": "lisa.doe@example.com",
    "documents_submitted": [
        "death_certificate", "policy_document", "medical_records",
        "will_document", "trust_document", "beneficiary_id", "police_report"
    ],
}

claim = CLAIM_SUBMISSION
docs = ", ".join(claim["documents_submitted"])

supervisor_prompt = (
    "## New Claim Submission\n\n"
    f"**Claim ID:** {claim['claim_id']}\n"
    f"**Policy Number:** {claim['policy_number']}\n"
    f"**Claimant Name:** {claim['claimant_name']}\n"
    f"**Date of Death:** {claim['date_of_death']}\n"
    f"**Death Circumstances:** {claim['death_circumstances']}\n"
    f"**Beneficiary Email:** {claim['beneficiary_email']}\n\n"
    f"### Documents Submitted\n{docs}\n\n"
    f"### S3 Location\nBucket: {S3_BUCKET}\nPrefix: {S3_PREFIX}\n\n"
    "Process this claim through the FULL pipeline: retrieve documents, run pre-processing, "
    "adjudicate, send notification, and persist the final record."
)

print("🚀 Starting End-to-End Pipeline")
print("=" * 60)
print(f"   Claim ID : {claim['claim_id']}")
print(f"   Policy   : {claim['policy_number']}")
print(f"   Claimant : {claim['claimant_name']}")
print("=" * 60)

with socotra_mcp_client:
    result = intake_orchestrator(supervisor_prompt)

print()
print("=" * 60)
print("✅ End-to-End Pipeline Complete")
print("=" * 60)

## Inspect Results

In [ ]:
print("📊 Pipeline Results")
print("=" * 60)
print()
print(str(result)[:3000])

## Read Back from DynamoDB

The claim record now includes all three phases — authentication, verification, adjudication decision, and notification status.

In [ ]:
table = dynamodb.Table(DYNAMODB_TABLE)
claim_id = CLAIM_SUBMISSION["claim_id"]

response_ddb = table.get_item(Key={"claim_id": claim_id})
item = response_ddb.get("Item", {})

if item:
    print(f"✅ Claim record retrieved")
    print(f"   Claim ID     : {item['claim_id']}")
    print(f"   Policy       : {item.get('policy_number', 'N/A')}")
    print(f"   Claimant     : {item.get('claimant_name', 'N/A')}")
    print(f"   Stage        : {item.get('stage', 'N/A')}")
    print(f"   Adjudication : {item.get('adjudication_decision', 'N/A')}")
    print(f"   Notification : {item.get('notification_status', 'N/A')}")
    print(f"   Pipeline     : {item.get('pipeline', 'N/A')}")
    print(f"   Created      : {item.get('created_at', 'N/A')}")
else:
    print(f"❌ Claim record not found for {claim_id}")

## ✅ Notebook Complete: End-to-End Claims Orchestration

You ran the full claim lifecycle in a single supervisor invocation:

| Phase | Tool | Result |
|---|---|---|
| Pre-Processing | `retrieve_claim_documents` | PDFs downloaded from S3 |
| Pre-Processing | `run_preprocessing_graph` | Authenticated, extracted, verified |
| Adjudication | `adjudicate_claim` | Auto-approved (all checks passed) |
| Communication | `send_claim_decision` | Notification drafted by Communicator Agent |
| Persistence | `persist_claim_to_dynamodb` | Full record written with `stage: pipeline_complete` |

### What About Claims That Need Human Review?

This notebook showed the happy path — a clean claim that auto-approves. In production, claims with flags, exclusions, or inconsistencies would be routed to a human adjudicator via a Step Functions callback workflow (or Amazon Quick Automate for case management and HITL task routing).

The optional **05_human_in_the_loop_integration** notebook demonstrates this escalation pattern using the same Intake Orchestrator with a modified `adjudicate_claim` tool that submits flagged claims to Step Functions instead of auto-approving.

## Cleanup (Optional)

Remove the local directories created during the pipeline run.

In [ ]:
import shutil

for folder in ['claim_documents', 'extracted_output']:
    if Path(folder).exists():
        shutil.rmtree(folder)
        print(f'✅ Removed {folder}/')
    else:
        print(f'ℹ️  {folder}/ not found (already clean)')